# Space Launch Mission Parameters Extraction

**Notebook 2c** - Data Cleaning Phase: Mission Parameters

## Purpose
Extract and clean mission-related parameters from raw Space Devs launch data, focusing on mission objectives, orbit details, and launch service provider information.

## Authors
- **Phillip Roman** - Mission parameter extraction and cleaning

## Workflow Position
1. Data Collection → Raw launch data collected via API
2. Data Cleaning → Extract parameters from raw data:
   - 2a. Extract rocket parameters
   - 2b. Extract launch parameters
   - 2c. **This Notebook** → Extract mission parameters (10 attributes)
3. Data Merging → Combine all cleaned datasets

## Key Parameters Extracted
- Mission identification and description
- Orbit specifications
- Program affiliations
- Launch service provider details

## Output
- `clean_mission_data.tsv` - Mission parameters ready for merging

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Required Libraries

Load necessary Python packages for data extraction and file operations.

In [2]:
import json
import pandas as pd
import csv
from pprint import pprint

## Load Raw Data File

**BEFORE USE:** Update filepath for raw_baseline_launches.json

### File Loading Strategy:
- **Error handling:** `try...except` catches missing file errors
- **Safe extraction:** `.get('launches', [])` returns empty list if key missing
- **Data source selection:** `USE_SAMPLE` flag controls dataset size

### USE_SAMPLE Flag:
- **False (current):** Process all 7,000+ launches for production
- **True:** Process last 50 launches for testing extraction logic

Sample flag allows quick check of extraction code before running on full dataset.

In [3]:
try:
    with open('/content/drive/MyDrive/DSCI511/Term Project/raw_baseline_launches_Phillip.json', 'r', encoding='utf-8') as f:
        full_data = json.load(f)
except FileNotFoundError:
    print("ERROR: raw_baseline_launches.json not found.")
    exit()

# list of all launches is inside the 'launches' key
all_launches = full_data.get('launches', [])
print(f"Loaded {len(all_launches)} total launches.")

USE_SAMPLE = False

if USE_SAMPLE:
    data_source = all_launches[-50:] # Sample last 50
else:
    data_source = all_launches # Full dataset

print(f"Processing {len(data_source)} launches...")

Loaded 7333 total launches.
Processing 7333 launches...


## Data Exploration and Validation (Optional)

The following two cells show the raw data structure and test extraction logic. Not required for the extraction workflow.

In [4]:
# check data structure and see all the keys
pprint(data_source[0])

{'agency_launch_attempt_count': 1,
 'agency_launch_attempt_count_year': 1,
 'failreason': '',
 'flightclub_url': None,
 'hashtag': None,
 'id': 'e3df2ecd-c239-472f-95e4-2b89b4f75800',
 'image': {'credit': None,
           'id': 1844,
           'image_url': 'https://thespacedevs-prod.nyc3.digitaloceanspaces.com/media/images/sputnik_8k74ps_image_20210830185541.jpg',
           'license': {'id': 1, 'link': None, 'name': 'Unknown', 'priority': 9},
           'name': '[AUTO] Sputnik 8K74PS - image',
           'single_use': True,
           'thumbnail_url': 'https://thespacedevs-prod.nyc3.digitaloceanspaces.com/media/images/255bauto255d__image_thumbnail_20240305193923.jpeg',
           'variants': []},
 'info_urls': [],
 'infographic': None,
 'last_updated': '2024-03-17T19:17:35Z',
 'launch_designator': '1957-001',
 'launch_service_provider': {'abbrev': 'CCCP',
                             'administrator': None,
                             'attempted_landings': 0,
                        

## Testing Extraction Logic - doing the Mapping
Mapping paths for 7 attributes on a single launch record before running the full loop on the sample.

Used following reference for safely unnesting dictionary using .get() chaining - a little experimenting:

 https://stackoverflow.com/questions/25833613/safe-method-to-get-value-of-nested-dictionary

In [5]:
test_launch = data_source[0]

print("Testing 'Mission' fields:")
print(f"Launch ID: {test_launch.get('id')}")
print(f"Mission Name: {test_launch.get('mission', {}).get('name')}")
print(f"Mission Type: {test_launch.get('mission', {}).get('type')}")
print(f"Orbit Name: {test_launch.get('mission', {}).get('orbit', {}).get('name')}")

print("\nTesting 'Crew/Program' field:")
print(f"Program: {test_launch.get('program')}") # shows it's a list

print("\nTesting 'Agency/LSP' fields:")
print(f"LSP Name: {test_launch.get('launch_service_provider', {}).get('name')}")
print(f"LSP Type: {test_launch.get('launch_service_provider', {}).get('type', {}).get('name')}")

Testing 'Mission' fields:
Launch ID: e3df2ecd-c239-472f-95e4-2b89b4f75800
Mission Name: Sputnik 1
Mission Type: Test Flight
Orbit Name: Low Earth Orbit

Testing 'Crew/Program' field:
Program: []

Testing 'Agency/LSP' fields:
LSP Name: Soviet Space Program
LSP Type: Government


## Mission Parameter Extraction

Extract 10 mission-related attributes from each launch record.

### Extraction Strategy:
- **Nested data handling:** Mission and orbit data require multiple `.get()` calls
- **Missing data:** if/else blocks provide None values for missing fields
- **Program data:** Currently extracts first program name (most launches have 0-1 programs)

### Parameters Extracted:
1. launch_id - Primary key for merging
2. mission_id - Mission identifier
3. mission_name - Mission designation
4. mission_type - Category (Test Flight, Communications, etc.)
5. mission_description - Detailed mission objective
6. orbit_name - Target orbit (Low Earth Orbit, GTO, etc.)
7. orbit_abbrev - Orbit abbreviation (LEO, GTO, etc.)
8. program_name - Associated program (ISS, Artemis, etc.)
9. lsp_name - Launch Service Provider
10. lsp_type - Provider type (Government, Commercial, etc.)

In [8]:
mission_table = []
mission_header = [
    "launch_id",
    "mission_id",
    "mission_name",
    "mission_type",
    "mission_description",
    "orbit_name",
    "orbit_abbrev",
    "program_name",
    "lsp_name",
    "lsp_type"
]

for launch in data_source:

    launch_id = launch.get('id') # primary attribute for merging

    # mission and orbit data
    mission_data = launch.get('mission')
    if mission_data:
        mission_id = mission_data.get('id')
        mission_name = mission_data.get('name')
        mission_type = mission_data.get('type')
        mission_desc = mission_data.get('description')

        # orbit data nested inside mission
        orbit_data = mission_data.get('orbit')
        if orbit_data:
            orbit_name = orbit_data.get('name')
            orbit_abbrev = orbit_data.get('abbrev')
        else:
            orbit_name = None
            orbit_abbrev = None
    else:
        # handle missing Mission data
        mission_id = None
        mission_name = None
        mission_type = None
        mission_desc = None
        orbit_name = None
        orbit_abbrev = None

    # extracts program data - looking for "crewed" indicator
    # stores as a list
    program_list = launch.get('program', [])
    if program_list:
        program_name = program_list[0].get('name') # first program only
    else:
        program_name = None

    # launch service provider data - aka "agency"
    lsp_data = launch.get('launch_service_provider')
    if lsp_data:
        lsp_name = lsp_data.get('name')

        # extract nested LSP type
        lsp_type_data = lsp_data.get('type')
        if lsp_type_data:
            lsp_type = lsp_type_data.get('name')
        else:
            lsp_type = None
    else:
        lsp_name = None
        lsp_type = None

    # builds the row for this launch
    row = [
        launch_id,
        mission_id,
        mission_name,
        mission_type,
        mission_desc,
        orbit_name,
        orbit_abbrev,
        program_name,
        lsp_name,
        lsp_type
    ]
    mission_table.append(row)

print(f"Finished mission parameter extraction. Created table with {len(mission_table)} rows.")

Finished mission parameter extraction. Created table with 7333 rows.


## Create DataFrame and Save Output

Convert extracted mission data to pandas DataFrame and export as TSV file.

### Data Quality Checks:
- Display value distributions for categorical fields
- Check for missing data using `.info()`
- Preview first rows to verify extraction

### Output Format:
TSV. Chosen over CSV since many attribute descriptions contain commas.

In [10]:
df_mission = pd.DataFrame(mission_table, columns=mission_header)
print("Successfully created Mission DataFrame:")

print("\nMission Type Breakdown:")
print(df_mission['mission_type'].value_counts())

print("\nLSP Type Breakdown:")
print(df_mission['lsp_type'].value_counts())

# preview DataFrame
print("\nPrinting first 3 rows:")
print(df_mission.head(3)) # displays plain text with labels

# data types and null counts
print("\nDataFrame Info (Checking for nulls)")
df_mission.info()

# Save options
# option 1 (default) - save to current directory
# output_path = ''

# option 2 (Colab users) - save to Google Drive
output_path = '/content/drive/MyDrive/DSCI511/Term Project/'

# option 3 - save to specific local path
# output_path = '/Users/YourName/Desktop/DSCI511/'

output_filename = 'clean_mission_data.tsv'

try:
    df_mission.to_csv(output_path + output_filename, sep='\t', index=False)
    print(f"Successfully saved mission data to {output_path + output_filename}")
    print(f"Shape: {df_mission.shape}")
except Exception as e:
    print(f"Error saving file: {e}")

Successfully created Mission DataFrame:

Mission Type Breakdown:
mission_type
Government/Top Secret          2253
Communications                 1568
Earth Science                   935
Navigation                      407
Test Flight                     342
Human Exploration               314
Test Target                     205
Astrophysics                    167
Resupply                        124
Lunar Exploration                88
Robotic Exploration              88
Dedicated Rideshare              67
Planetary Science                59
Technology                       43
Heliophysics                     37
                                 26
Tourism                          25
Materials Science                24
Suborbital                       22
Biology                          18
Unknown                           3
Space Situational Awareness       1
Name: count, dtype: int64

LSP Type Breakdown:
lsp_type
Government       5307
Commercial       1942
Private            72
Multinat

## (Optional) Preview First 10 Rows

Displaying direct as HTML table for better readability.

In [ ]:
df_mission.head(10)

,launch_id,mission_id,mission_name,mission_type,mission_description,orbit_name,orbit_abbrev,program_name,lsp_name,lsp_type
0,e3df2ecd-c239-472f-95e4-2b89b4f75800,1430.0,Sputnik 1,Test Flight,First artificial satellite consisting of a 58 ...,Low Earth Orbit,LEO,None,Soviet Space Program,Government
1,f8c9f344-a6df-4f30-873a-90fe3a7840b3,1431.0,Sputnik 2,Test Flight,Second artificial satellite and first to carry...,Low Earth Orbit,LEO,None,Soviet Space Program,Government
2,535c1a09-97c8-4f96-bb64-6336d4bcb1fb,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
3,1b9e28d0-c531-44b0-9b37-244e62a6d3f4,1433.0,Explorer 1,Test Flight,First successfully launched American satellite...,Low Earth Orbit,LEO,None,Army Ballistic Missile Agency,Government
4,48bc7deb-b2e1-46c2-ab63-0ce00fbd192b,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
5,896e8af6-d256-4a5b-ab15-2f25c84e90e3,1434.0,Explorer 2,Test Flight,Small satellite similar to Explorer 1. It fail...,Low Earth Orbit,LEO,None,Army Ballistic Missile Agency,Government
6,74d39bb8-34a6-4a8b-8554-d2d3ec22aee6,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
7,b4e501ff-083c-47d6-9ff0-63ec1bf035c3,1435.0,Explorer 3,Earth Science,Small satellite launched into an eccentric orb...,Elliptical Orbit,Elliptical,None,Army Ballistic Missile Agency,Government
8,59d2de37-4c22-495f-8718-4b22f5f34ab7,1436.0,D-1 1,Earth Science,First complex scientific satellite with 12 exp...,Low Earth Orbit,LEO,None,Soviet Space Program,Government
9,de282e74-e03b-411e-9633-2d1497629893,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
